In [1]:
import tkinter as tk
from tkinter import messagebox, ttk
import time

# =========================
# QUIXO 5x5 — Single file
# Direction buttons mean: "INSERT FROM <SIDE>"
# (This matches real Quixo and fixes middle-not-filling behavior.)
# =========================

N = 5
EMPTY = " "
HUMAN = "X"
AI = "O"

REPEAT_LIMIT = 3
MOVE_LIMIT = 200

AI_TIME_LIMIT_SEC = 1.0
AI_MAX_DEPTH = 5


def is_border(r, c):
    return r == 0 or r == N - 1 or c == 0 or c == N - 1


def board_to_tuple(b):
    return tuple(tuple(row) for row in b)


def tuple_to_board(t):
    return [list(row) for row in t]


def check_winner(b):
    lines = []
    for r in range(N):
        lines.append(b[r])
    for c in range(N):
        lines.append([b[r][c] for r in range(N)])
    lines.append([b[i][i] for i in range(N)])
    lines.append([b[i][N - 1 - i] for i in range(N)])

    x_win = any(all(v == HUMAN for v in line) for line in lines)
    o_win = any(all(v == AI for v in line) for line in lines)

    if x_win and o_win:
        return "BOTH"
    if x_win:
        return HUMAN
    if o_win:
        return AI
    return None


def allowed_insert_sides_for_pick(r, c):
    """
    Valid "insert from side" choices for a picked border cube.
    Rule: you cannot insert from the SAME side you removed from (outward),
    and corners have only 2 inward options.
    """
    if (r, c) == (0, 0):            # top-left removed from TOP/LEFT -> can insert from RIGHT or DOWN
        return ["RIGHT", "DOWN"]
    if (r, c) == (0, N - 1):        # top-right -> insert from LEFT or DOWN
        return ["LEFT", "DOWN"]
    if (r, c) == (N - 1, 0):        # bottom-left -> insert from RIGHT or UP
        return ["RIGHT", "UP"]
    if (r, c) == (N - 1, N - 1):    # bottom-right -> insert from LEFT or UP
        return ["LEFT", "UP"]

    # edges (non-corner)
    if r == 0:            # top edge: can't insert from UP
        return ["LEFT", "RIGHT", "DOWN"]
    if r == N - 1:        # bottom edge: can't insert from DOWN
        return ["LEFT", "RIGHT", "UP"]
    if c == 0:            # left edge: can't insert from LEFT
        return ["UP", "DOWN", "RIGHT"]
    if c == N - 1:        # right edge: can't insert from RIGHT
        return ["UP", "DOWN", "LEFT"]

    return []


def apply_move_quixo(b, r, c, insert_from, player):
    """
    Quixo-correct move:
    - Pick border cube at (r,c), must be EMPTY or player's symbol.
    - Remove it (create gap).
    - Insert player's cube from chosen side (insert_from),
      shifting the line and filling the gap properly.
    """
    if not is_border(r, c):
        raise ValueError("Pick must be on the border.")
    if b[r][c] not in (EMPTY, player):
        raise ValueError("Pick must be EMPTY or your own symbol.")
    allowed = allowed_insert_sides_for_pick(r, c)
    if insert_from not in allowed:
        raise ValueError(f"Invalid direction. Allowed: {allowed}")

    nb = [row[:] for row in b]

    # If insert_from is LEFT/RIGHT => operate on ROW r
    if insert_from in ("LEFT", "RIGHT"):
        row = nb[r][:]

        picked = row.pop(c)  # remove selected cube at its actual location

        if insert_from == "LEFT":
            # Insert at left end, push toward RIGHT.
            # Gap is filled by shifting elements between left end and gap.
            row.insert(0, player)
            # Now row has length N again
        else:
            # Insert at right end, push toward LEFT.
            row.append(player)

        # row currently length N (after one insert)
        # BUT: we must ensure we shifted correctly relative to where the gap was.
        # The pop(c) already removed the selected cube.
        # Inserting at an end shifts the correct segment automatically in Python list.
        # This is the correct Quixo behavior.

        nb[r] = row
        return nb

    # If insert_from is UP/DOWN => operate on COLUMN c
    col = [nb[i][c] for i in range(N)]
    picked = col.pop(r)

    if insert_from == "UP":
        # Insert at top, push DOWN.
        col.insert(0, player)
    else:  # DOWN
        # Insert at bottom, push UP.
        col.append(player)

    for i in range(N):
        nb[i][c] = col[i]
    return nb


def legal_moves(b, player):
    moves = []
    for r in range(N):
        for c in range(N):
            if not is_border(r, c):
                continue
            if b[r][c] not in (EMPTY, player):
                continue
            for ins in allowed_insert_sides_for_pick(r, c):
                moves.append((r, c, ins))
    return moves


# ---------------- AI (Minimax + Alpha-Beta) ----------------
def evaluate_board(b):
    w = check_winner(b)
    if w == AI:
        return 10_000
    if w == HUMAN:
        return -10_000
    if w == "BOTH":
        return 0

    score = 0
    lines = []
    for r in range(N):
        lines.append(b[r])
    for c in range(N):
        lines.append([b[r][c] for r in range(N)])
    lines.append([b[i][i] for i in range(N)])
    lines.append([b[i][N - 1 - i] for i in range(N)])

    for line in lines:
        x = sum(1 for v in line if v == HUMAN)
        o = sum(1 for v in line if v == AI)
        if x > 0 and o > 0:
            continue
        if o > 0:
            score += o * o
        elif x > 0:
            score -= x * x

    # small center bonus
    if b[2][2] == AI:
        score += 1
    elif b[2][2] == HUMAN:
        score -= 1

    return score


def opponent(p):
    return HUMAN if p == AI else AI


def minimax(board_t, player, depth, alpha, beta, tt):
    key = (board_t, player, depth)
    if key in tt:
        return tt[key]

    b = tuple_to_board(board_t)
    w = check_winner(b)
    if w is not None or depth == 0:
        out = (evaluate_board(b), None)
        tt[key] = out
        return out

    moves = legal_moves(b, player)
    if not moves:
        # treat as losing for current player
        out = (-10_000 if player == AI else 10_000, None)
        tt[key] = out
        return out

    # ordering
    def mscore(m):
        nb = apply_move_quixo(b, *m, player)
        w2 = check_winner(nb)
        if w2 == player or w2 == "BOTH":
            return 20_000
        return evaluate_board(nb)

    moves.sort(key=mscore, reverse=(player == AI))

    best_move = None

    if player == AI:
        best_val = -10**18
        for m in moves:
            nb = apply_move_quixo(b, *m, player)
            val, _ = minimax(board_to_tuple(nb), opponent(player), depth - 1, alpha, beta, tt)
            if val > best_val:
                best_val, best_move = val, m
            alpha = max(alpha, best_val)
            if beta <= alpha:
                break
        out = (best_val, best_move)
    else:
        best_val = 10**18
        for m in moves:
            nb = apply_move_quixo(b, *m, player)
            val, _ = minimax(board_to_tuple(nb), opponent(player), depth - 1, alpha, beta, tt)
            if val < best_val:
                best_val, best_move = val, m
            beta = min(beta, best_val)
            if beta <= alpha:
                break
        out = (best_val, best_move)

    tt[key] = out
    return out


def ai_best_move(board, time_limit=AI_TIME_LIMIT_SEC, max_depth=AI_MAX_DEPTH):
    start = time.time()
    best = None
    board_t = board_to_tuple(board)
    tt = {}

    for d in range(1, max_depth + 1):
        if time.time() - start > time_limit:
            break
        val, mv = minimax(board_t, AI, d, -10**18, 10**18, tt)
        if time.time() - start > time_limit:
            break
        if mv is not None:
            best = mv

    if best is None:
        ms = legal_moves(board, AI)
        best = ms[0] if ms else None
    return best


# ---------------- GUI ----------------
class QuixoGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("Quixo 5x5 — Human (X) vs AI (O)")

        self.board = [[EMPTY] * N for _ in range(N)]
        self.turn = HUMAN
        self.selected = None
        self.move_count = 0
        self.rep = {}

        top = tk.Frame(root)
        top.pack(padx=10, pady=6, fill="x")

        self.status = tk.StringVar(value="Your turn: pick a BORDER cube (empty or X).")
        tk.Label(top, textvariable=self.status, fg="blue").pack(side="left")

        tk.Button(top, text="New Game", command=self.new_game).pack(side="right")

        mid = tk.Frame(root)
        mid.pack(padx=10, pady=6)

        self.buttons = [[None] * N for _ in range(N)]
        for r in range(N):
            for c in range(N):
                btn = tk.Button(
                    mid, text=" ", width=4, height=2,
                    command=lambda rr=r, cc=c: self.on_cell(rr, cc),
                    font=("Arial", 18, "bold")
                )
                btn.grid(row=r, column=c, padx=2, pady=2)
                self.buttons[r][c] = btn

        dframe = tk.Frame(root)
        dframe.pack(padx=10, pady=6)

        self.dir_buttons = {}
        for d in ["UP", "DOWN", "LEFT", "RIGHT"]:
            b = tk.Button(dframe, text=d, width=8, state="disabled",
                          command=lambda dd=d: self.on_direction(dd))
            b.pack(side="left", padx=4)
            self.dir_buttons[d] = b

        sframe = tk.Frame(root)
        sframe.pack(padx=10, pady=(0, 10), fill="x")

        tk.Label(sframe, text="AI time (sec):").pack(side="left")
        self.ai_time_var = tk.DoubleVar(value=AI_TIME_LIMIT_SEC)
        ttk.Scale(sframe, from_=0.3, to=2.0, orient="horizontal",
                  length=180, variable=self.ai_time_var).pack(side="left", padx=8)

        tk.Label(sframe, text="AI max depth:").pack(side="left", padx=(12, 0))
        self.ai_depth_var = tk.IntVar(value=AI_MAX_DEPTH)
        ttk.Scale(sframe, from_=2, to=6, orient="horizontal",
                  length=160, variable=self.ai_depth_var).pack(side="left", padx=8)

        self.refresh()

    def new_game(self):
        self.board = [[EMPTY] * N for _ in range(N)]
        self.turn = HUMAN
        self.selected = None
        self.move_count = 0
        self.rep = {}
        self.disable_dirs()
        self.status.set("Your turn: pick a BORDER cube (empty or X).")
        self.refresh()

    def refresh(self):
        for r in range(N):
            for c in range(N):
                self.buttons[r][c].config(text=self.board[r][c])
                self.buttons[r][c].config(relief=("sunken" if self.selected == (r, c) else "raised"))

    def disable_dirs(self):
        for b in self.dir_buttons.values():
            b.config(state="disabled")

    def enable_dirs(self, dirs):
        for d, b in self.dir_buttons.items():
            b.config(state=("normal" if d in dirs else "disabled"))

    def rep_touch(self):
        key = (board_to_tuple(self.board), self.turn)
        self.rep[key] = self.rep.get(key, 0) + 1
        return self.rep[key] >= REPEAT_LIMIT

    def end_check(self, mover):
        w = check_winner(self.board)
        if w == "BOTH":
            w = mover  # mover gets credit

        if w == HUMAN:
            messagebox.showinfo("Game Over", "You win! 🎉")
            return True
        if w == AI:
            messagebox.showinfo("Game Over", "AI wins! 💀")
            return True

        self.move_count += 1
        if self.move_count >= MOVE_LIMIT:
            messagebox.showinfo("Game Over", "Draw (move limit).")
            return True

        if self.rep_touch():
            messagebox.showinfo("Game Over", "Draw (3-fold repetition).")
            return True

        return False

    def on_cell(self, r, c):
        if self.turn != HUMAN:
            return

        if not is_border(r, c):
            self.status.set("Pick a BORDER cube.")
            return
        if self.board[r][c] not in (EMPTY, HUMAN):
            self.status.set("You can pick only EMPTY or your own (X) cube.")
            return

        self.selected = (r, c)
        dirs = allowed_insert_sides_for_pick(r, c)
        self.enable_dirs(dirs)
        self.status.set(f"Selected ({r},{c}). INSERT FROM: {', '.join(dirs)}.")
        self.refresh()

    def on_direction(self, insert_from):
        if self.turn != HUMAN or self.selected is None:
            return

        r, c = self.selected
        try:
            self.board = apply_move_quixo(self.board, r, c, insert_from, HUMAN)
        except Exception as e:
            self.status.set(str(e))
            return

        self.selected = None
        self.disable_dirs()
        self.refresh()

        if self.end_check(mover=HUMAN):
            return

        self.turn = AI
        self.status.set("AI thinking...")
        self.root.after(20, self.ai_play)

    def ai_play(self):
        t = float(self.ai_time_var.get())
        d = int(round(self.ai_depth_var.get()))

        move = ai_best_move(self.board, time_limit=t, max_depth=d)
        if move is None:
            messagebox.showinfo("Game Over", "Draw (no moves).")
            return

        r, c, ins = move
        self.board = apply_move_quixo(self.board, r, c, ins, AI)
        self.refresh()

        if self.end_check(mover=AI):
            return

        self.turn = HUMAN
        self.status.set("Your turn: pick a BORDER cube (empty or X).")


if __name__ == "__main__":
    root = tk.Tk()
    QuixoGUI(root)
    root.mainloop()